In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy import stats

In [2]:
def createdata():
  data = {
      'Age': np.random.randint(18, 70, size=20),
      'Salary': np.random.randint(30000, 120000, size=20),
      'Purchased': np.random.choice([0, 1], size=20),
      'Gender': np.random.choice(['Male', 'Female'], size=20),
      'City': np.random.choice(['New York', 'San Francisco', 'Los Angeles'], size=20)
  }

  df = pd.DataFrame(data)
  return df

In [3]:
df = createdata()
df.head(10)

,Age,Salary,Purchased,Gender,City
0,31,72779,1,Male,San Francisco
1,31,68565,1,Male,San Francisco
2,59,88399,1,Male,San Francisco
3,21,33182,0,Female,San Francisco
4,38,58152,1,Female,San Francisco
5,63,104883,0,Female,San Francisco
6,43,96859,0,Male,Los Angeles
7,20,109709,0,Female,San Francisco
8,62,82158,1,Male,San Francisco
9,28,99534,1,Male,San Francisco


In [4]:
df.shape

(20, 5)

In [6]:
# Introduce some missing values for demonstration
df.loc[5, 'Age'] = np.nan
df.loc[10, 'Salary'] = np.nan
df.head(10)

,Age,Salary,Purchased,Gender,City
0,31.0,72779.0,1,Male,San Francisco
1,31.0,68565.0,1,Male,San Francisco
2,59.0,88399.0,1,Male,San Francisco
3,21.0,33182.0,0,Female,San Francisco
4,38.0,58152.0,1,Female,San Francisco
5,NaN,104883.0,0,Female,San Francisco
6,43.0,96859.0,0,Male,Los Angeles
7,20.0,109709.0,0,Female,San Francisco
8,62.0,82158.0,1,Male,San Francisco
9,28.0,99534.0,1,Male,San Francisco


In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Age        19 non-null     float64
 1   Salary     19 non-null     float64
 2   Purchased  20 non-null     int64  
 3   Gender     20 non-null     object 
 4   City       20 non-null     object 
dtypes: float64(2), int64(1), object(2)
memory usage: 932.0+ bytes
None


In [8]:
print(df.describe())

             Age         Salary  Purchased
count  19.000000      19.000000  20.000000
mean   39.631579   76143.631579   0.500000
std    14.407600   24171.475051   0.512989
min    18.000000   33182.000000   0.000000
25%    28.000000   59244.500000   0.000000
50%    38.000000   77331.000000   0.500000
75%    53.000000   95815.500000   1.000000
max    62.000000  112452.000000   1.000000


In [9]:
# Check for missing values in each column
missing_values = df.isnull().sum()

# Display columns with missing values
print(missing_values[missing_values > 0])

Age       1
Salary    1
dtype: int64


In [10]:
#Create an instance of SimpleImputer with the median strategy for Age and mean stratergy for Salary
imputer1 = SimpleImputer(strategy="median")
imputer2 = SimpleImputer(strategy="mean")

df_copy=df

#Fit the imputer on the "Age" and "Salary"column

imputer1.fit(df_copy[["Age"]])
imputer2.fit(df_copy[["Salary"]])

#Transform (fill) the missing values in the "Age" and "Salary"c column
df_copy["Age"] = imputer1.transform(df[["Age"]])
df_copy["Salary"] = imputer2.transform(df[["Salary"]])

# Verify that there are no missing values left
print(df_copy["Age"].isnull().sum())
print(df_copy["Salary"].isnull().sum())

0
0


In [11]:
#Handling Categorical Attributes
#Using Ordinal Encoding for gender COlumn and One-Hot Encoding for City Column

# Initialize OrdinalEncoder
ordinal_encoder = OrdinalEncoder(categories=[["Male", "Female"]])
# Fit and transform the data
df_copy["Gender_Encoded"] = ordinal_encoder.fit_transform(df_copy[["Gender"]])

# Initialize OneHotEncoder
onehot_encoder = OneHotEncoder()

# Fit and transform the "City" column
encoded_data = onehot_encoder.fit_transform(df[["City"]])

# Convert the sparse matrix to a dense array
encoded_array = encoded_data.toarray()

# Convert to DataFrame for better visualization
encoded_df = pd.DataFrame(encoded_array, columns=onehot_encoder.get_feature_names_out(["City"]))
df_encoded = pd.concat([df_copy, encoded_df], axis=1)

df_encoded.drop("Gender", axis=1, inplace=True)
df_encoded.drop("City", axis=1, inplace=True)

print(df_encoded. head())

    Age   Salary  Purchased  Gender_Encoded  City_Los Angeles  City_New York  \
0  31.0  72779.0          1             0.0               0.0            0.0   
1  31.0  68565.0          1             0.0               0.0            0.0   
2  59.0  88399.0          1             0.0               0.0            0.0   
3  21.0  33182.0          0             1.0               0.0            0.0   
4  38.0  58152.0          1             1.0               0.0            0.0   

   City_San Francisco  
0                 1.0  
1                 1.0  
2                 1.0  
3                 1.0  
4                 1.0  


In [12]:
#Data Transformation
# Min-Max Scaler/Normalization (range 0-1)
normalizer = MinMaxScaler()
df_encoded[['Salary']] = normalizer.fit_transform(df_encoded[['Salary']])
df_encoded.head()

,Age,Salary,Purchased,Gender_Encoded,City_Los Angeles,City_New York,City_San Francisco
0,31.0,0.499521,1,0.0,0.0,0.0,1.0
1,31.0,0.446361,1,0.0,0.0,0.0,1.0
2,59.0,0.696569,1,0.0,0.0,0.0,1.0
3,21.0,0.000000,0,1.0,0.0,0.0,1.0
4,38.0,0.314999,1,1.0,0.0,0.0,1.0


In [13]:
# Standardization (mean=0, variance=1)
scaler = StandardScaler()
df_encoded[['Age']] = scaler.fit_transform(df_encoded[['Age']])
df_encoded.head()


,Age,Salary,Purchased,Gender_Encoded,City_Los Angeles,City_New York,City_San Francisco
0,-0.625326,0.499521,1,0.0,0.0,0.0,1.0
1,-0.625326,0.446361,1,0.0,0.0,0.0,1.0
2,1.422525,0.696569,1,0.0,0.0,0.0,1.0
3,-1.356701,0.000000,0,1.0,0.0,0.0,1.0
4,-0.113363,0.314999,1,1.0,0.0,0.0,1.0


In [14]:
# Outlier Detection and Treatment using IQR
df_encoded_copy1=df_encoded
df_encoded_copy2=df_encoded
df_encoded_copy3=df_encoded

Q1 = df_encoded_copy1['Salary'].quantile(0.25)
Q3 = df_encoded_copy1['Salary'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df_encoded_copy1['Salary'] = np.where(df_encoded_copy1['Salary'] > upper_bound, upper_bound,
                        np.where(df_encoded_copy1['Salary'] < lower_bound, lower_bound, df_encoded_copy1['Salary']))

print(df_encoded_copy1.head())



        Age    Salary  Purchased  Gender_Encoded  City_Los Angeles  \
0 -0.625326  0.499521          1             0.0               0.0   
1 -0.625326  0.446361          1             0.0               0.0   
2  1.422525  0.696569          1             0.0               0.0   
3 -1.356701  0.000000          0             1.0               0.0   
4 -0.113363  0.314999          1             1.0               0.0   

   City_New York  City_San Francisco  
0            0.0                 1.0  
1            0.0                 1.0  
2            0.0                 1.0  
3            0.0                 1.0  
4            0.0                 1.0  


In [15]:
df_encoded_copy2['Salary_zscore'] = stats.zscore(df_encoded_copy2['Salary'])
df_encoded_copy2['Salary'] = np.where(df_encoded_copy2['Salary_zscore'].abs() > 3, np.nan, df_encoded_copy2['Salary'])  # Replace outliers with NaN
print(df_encoded_copy2.head())

        Age    Salary  Purchased  Gender_Encoded  City_Los Angeles  \
0 -0.625326  0.499521          1             0.0               0.0   
1 -0.625326  0.446361          1             0.0               0.0   
2  1.422525  0.696569          1             0.0               0.0   
3 -1.356701  0.000000          0             1.0               0.0   
4 -0.113363  0.314999          1             1.0               0.0   

   City_New York  City_San Francisco  Salary_zscore  
0            0.0                 1.0      -0.146728  
1            0.0                 1.0      -0.330496  
2            0.0                 1.0       0.534444  
3            0.0                 1.0      -1.873511  
4            0.0                 1.0      -0.784596  


In [16]:
df_encoded_copy3['Salary_zscore'] = stats.zscore(df_encoded_copy3['Salary'])
median_salary = df_encoded_copy3['Salary'].median()
df_encoded_copy3['Salary'] = np.where(df_encoded_copy3['Salary_zscore'].abs() > 3, median_salary, df_encoded_copy3['Salary'])
print(df_encoded_copy3.head())

        Age    Salary  Purchased  Gender_Encoded  City_Los Angeles  \
0 -0.625326  0.499521          1             0.0               0.0   
1 -0.625326  0.446361          1             0.0               0.0   
2  1.422525  0.696569          1             0.0               0.0   
3 -1.356701  0.000000          0             1.0               0.0   
4 -0.113363  0.314999          1             1.0               0.0   

   City_New York  City_San Francisco  Salary_zscore  
0            0.0                 1.0      -0.146728  
1            0.0                 1.0      -0.330496  
2            0.0                 1.0       0.534444  
3            0.0                 1.0      -1.873511  
4            0.0                 1.0      -0.784596  
